# Submission Notebook

This notebook defines the `measure_z12` and `measure_z34` stabilizer-measurement helpers, draws an example circuit, and verifies the measurement outcomes on a simulator.

## Dependencies

The `QiskitCode` environment on this computer already has the required packages installed:

- `qiskit`
- `qiskit-aer`
- `ipykernel`
- `jupyter`

If you ever need to install them again in a notebook, you can run:

```python
%pip install qiskit qiskit-aer ipykernel jupyter
```

In [20]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from IPython.display import display

In [6]:
from qiskit import QuantumCircuit

def measure_X(qc, ancilla=4, cbit=0, reset_ancilla=False):
    qc.h(ancilla)
    qc.cx(ancilla, 0)
    qc.cx(ancilla, 1)
    qc.cx(ancilla, 2)
    qc.cx(ancilla, 3)
    qc.h(ancilla)
    qc.measure(ancilla, cbit)
    if reset_ancilla:
        qc.reset(ancilla)

qc = QuantumCircuit(5, 1)
measure_X(qc)
print(qc.draw("text"))

ModuleNotFoundError: No module named 'qiskit'

In [21]:
def measure_z12(qc, ancilla=4, cbit=None, reset_ancilla=False):
    qc.h(ancilla)
    qc.cz(ancilla, 1)
    qc.cz(ancilla, 0)
    qc.h(ancilla)

    # If no classical bit is provided, store the result in c0.
    if cbit is None:
        cbit = 0

    qc.measure(ancilla, cbit)

    if reset_ancilla:
        qc.reset(ancilla)

    return qc


def measure_z34(qc, ancilla=4, cbit=None, reset_ancilla=False):
    qc.h(ancilla)
    qc.cz(ancilla, 3)
    qc.cz(ancilla, 2)
    qc.h(ancilla)

    if cbit is None:
        cbit = 0

    qc.measure(ancilla, cbit)

    if reset_ancilla:
        qc.reset(ancilla)

    return qc

In [22]:
# Example: prepare the first four qubits in |0010>.
# Here q0=0, q1=0, q2=1, q3=0.
# So Z12 should give 0 and Z34 should give 1.
example_qc = QuantumCircuit(5, 2)
example_qc.x(2)

measure_z12(example_qc, ancilla=4, cbit=0, reset_ancilla=True)
measure_z34(example_qc, ancilla=4, cbit=1)

display(example_qc.draw("text"))

q_0: ─────────■─────────────────────────────────
              │                                 
q_1: ──────■──┼─────────────────────────────────
     ┌───┐ │  │                                 
q_2: ┤ X ├─┼──┼───────────────────────■─────────
     └───┘ │  │                       │         
q_3: ──────┼──┼────────────────────■──┼─────────
     ┌───┐ │  │ ┌───┐┌─┐     ┌───┐ │  │ ┌───┐┌─┐
q_4: ┤ H ├─■──■─┤ H ├┤M├─|0>─┤ H ├─■──■─┤ H ├┤M├
     └───┘      └───┘└╥┘     └───┘      └───┘└╥┘
c: 2/═════════════════╩═══════════════════════╩═
                      0                       1

In [ ]:
def prepare_basis_state(qc, bits):
    """Prepare qubits q0, q1, q2, q3 from a tuple like (0, 1, 1, 0)."""
    for qubit, bit in enumerate(bits):
        if bit:
            qc.x(qubit)


def run_stabilizer_measurements(bits, shots=256):
    qc = QuantumCircuit(5, 2)
    prepare_basis_state(qc, bits)

    measure_z12(qc, ancilla=4, cbit=0, reset_ancilla=True)
    measure_z34(qc, ancilla=4, cbit=1)

    simulator = AerSimulator()
    compiled = transpile(qc, simulator)
    result = simulator.run(compiled, shots=shots).result()
    counts = result.get_counts()
    return qc, counts


test_cases = [
    (0, 0, 0, 0),
    (1, 0, 0, 0),
    (0, 0, 1, 0),
    (1, 1, 0, 1),
]

for bits in test_cases:
    qc, counts = run_stabilizer_measurements(bits)
    expected_z12 = bits[0] ^ bits[1]
    expected_z34 = bits[2] ^ bits[3]
    expected_bitstring = f"{expected_z34}{expected_z12}"

    print(f"data state q0,q1,q2,q3 = {bits}")
    print(f"counts = {counts}")
    print(f"expected dominant bitstring = {expected_bitstring}\n")

    assert list(counts.keys()) == [expected_bitstring], (
        f"Expected only {expected_bitstring}, got {counts}"
    )

print("All simulator tests passed.")
print("Reminder: Qiskit prints classical bitstrings as c1c0, so Z34 appears before Z12.")